# 5. Tables and figures
Summary tables, all figures of the paper, and the comparison of the new results with the archived ones.

Run the notebooks 1 to 4 first. The figures use the font Liberation Sans if it is installed (`apt-get install -y fonts-liberation`), otherwise DejaVu Sans.

In [ ]:
import torch
import pandas as pd

from constants import *
from utils import get_device
from dataset import load_pieces, get_duration_values
from disk_utils import load_results
from faithfulness import load_deletion_tests
from attention_inspector import AttentionInspector
from plot_manager import PlotManager
from result_tables import (
    per_seed_table,
    summary_table,
    paired_tests,
    pooled_report,
    list_confusions,
    group_errors,
    write_summary_files
)
from compare_with_archive import (
    compare_classification_results,
    summarize_comparison,
    compare_json_numbers,
    compare_csv_numbers
)

In [ ]:
USE_GPU = True
device = get_device(USE_GPU)
print(device)

# QUICK_TEST = True reads the dry run in results_quick_test/ and writes
# the figures to figs_quick_test/.
QUICK_TEST = False

if QUICK_TEST:
    results_dir = QUICK_TEST_RESULTS_DIR
    figs_dir = QUICK_TEST_FIGS_DIR
else:
    results_dir = RESULTS_DIR
    figs_dir = FIGS_DIR

# set to False to look at the figures without writing the files
SAVE_FIGS = True

## Tables

All accuracies as text files: `summary.txt`, `full_pooled.txt`, `paired_tests.txt`.

In [ ]:
write_summary_files(results_dir)

Mean ± 95% CI over the five splits (Tables 4 and 7).

In [ ]:
results = load_results(results_dir)
per_seed = per_seed_table(results)
summary = summary_table(per_seed)

pd.set_option("display.max_rows", 200)
summary[["tag", "perturbation", "n", "acc_mean", "acc_ci", "f1_mean", "f1_ci"]]

Paired comparison with the proposed model (the Δ and p columns of Tables 4 and 7).

In [ ]:
paired_tests(per_seed, reference_tag="full")

Per-makam precision, recall and F1 of the proposed model, pooled over the five test subsets (Table 5).

In [ ]:
# the unperturbed runs of the proposed model
main_results = []
for result in load_results(results_dir, "full"):
    if not result.get("perturbation"):
        main_results.append(result)

report, cm = pooled_report(main_results)
report.round(2)

Confusions of the proposed model, most frequent first (Table 6).

In [ ]:
for true_makam, predicted_makam, count in list_confusions(cm):
    if count >= 2:
        print(f"{true_makam:>16s} -> {predicted_makam:<16s} {count}")

Errors inside the group Uşşak–Hüseyni–Muhayyer and between Segah and Hüzzam, for every model.

In [ ]:
group_errors(results)

## Figures

In [ ]:
pm = PlotManager(figs_dir=figs_dir)

pieces, labels, vocab = load_pieces()
dur_values = get_duration_values(vocab)

Figure 1: pieces per makam

In [ ]:
pm.plot_makam_distribution(labels, save_fig=SAVE_FIGS)

Figure 2: notes per piece

In [ ]:
pm.plot_length_histogram(pieces, save_fig=SAVE_FIGS)

Figure 4: overview of all accuracies

In [ ]:
pm.plot_results_overview(summary, save_fig=SAVE_FIGS)

Figure 5: confusion matrix of the proposed model

In [ ]:
pm.plot_confusion_matrix(cm.values, save_fig=SAVE_FIGS)

Figure 6: deletion tests

In [ ]:
curves, deletion_summary = load_deletion_tests("full", results_dir)
pm.plot_deletion_curves(curves, deletion_summary, tag="full", save_fig=SAVE_FIGS)

Figure 7 and Figure C1: pitch enrichment per makam, ordered by frequency and by pitch

In [ ]:
ai = AttentionInspector(device, results_dir=results_dir)

note_table = ai.load_note_table("full")
enrichment = ai.pitch_enrichment(note_table)

In [ ]:
pm.plot_pitch_enrichment(enrichment, tag="full", pitch_order=False, save_fig=SAVE_FIGS)

In [ ]:
pm.plot_pitch_enrichment(enrichment, tag="full", pitch_order=True, save_fig=SAVE_FIGS)

Figure 8: attention over one test piece

In [ ]:
piece, attn, predicted_makam = ai.get_piece_attention(
    tag="full",
    seed=EXAMPLE_PIECE_SEED,
    piece_name=EXAMPLE_PIECE
)
print(piece["f_name"], "| notes:", len(attn), "| predicted:", predicted_makam)

pm.plot_example_piece(
    piece["notes"],
    attn,
    dur_values,
    ai.spelling_map,
    save_fig=SAVE_FIGS
)

Figure 9: attention by relative duration (rests left out)

In [ ]:
pitched_notes = note_table[~note_table["is_rest"]]
unit_by_duration = ai.unit_by_duration_bin(pitched_notes, "norm_attn")

pm.plot_attention_by_duration(unit_by_duration, tag="full", save_fig=SAVE_FIGS)

Figure 10: attention by position in the measure, for the proposed model and for the model without the measure-position input

In [ ]:
unit_by_position = ai.unit_by_measure_position(note_table, "norm_attn")
pm.plot_attention_by_measure_position(unit_by_position, tag="full", save_fig=SAVE_FIGS)

In [ ]:
note_table_no_meas = ai.load_note_table("no_meas")
unit_by_position_no_meas = ai.unit_by_measure_position(note_table_no_meas, "norm_attn")
pm.plot_attention_by_measure_position(unit_by_position_no_meas, tag="no_meas", save_fig=SAVE_FIGS)

## Comparison with the archived results
`results_archive/` holds the results of the previous run. On the same GPU type and library versions the new run should reproduce them exactly. On other hardware small differences are expected.

Classification results: are the predictions of every run the same?

In [ ]:
comparison = compare_classification_results(results_dir, ARCHIVE_RESULTS_DIR)
summarize_comparison(comparison)

Hyperparameter search, deletion tests and attention statistics: which numbers differ? (The `seconds` column of the hyperparameter search is the training time and always differs.)

In [ ]:
compare_csv_numbers("hparam_search.csv", results_dir, ARCHIVE_RESULTS_DIR)

In [ ]:
compare_csv_numbers("faithfulness/full_summary.csv", results_dir, ARCHIVE_RESULTS_DIR)

In [ ]:
compare_json_numbers("attention/full_report.json", results_dir, ARCHIVE_RESULTS_DIR)

In [ ]:
compare_json_numbers("attention/no_meas_report.json", results_dir, ARCHIVE_RESULTS_DIR)